# 02: Data Cleaning & Preparation

This notebook transforms the raw USA Real Estate dataset into a clean,
modelling-ready version. Building on insights from `01_eda`, we will:

- Re-inspect structure and missingness with a cleaning focus.
- Auto-derive imputation strategies using feature metadata.
- Perform *advanced numeric imputation* with IterativeImputer +
  ExtraTreesRegressor and KNNImputer where appropriate.
- Detect outliers via:
  - **Univariate methods** (Z-Score, IQR, MAD; chosen automatically).
  - **Multivariate method** (Isolation Forest).
- Optionally cap or remove outliers, while preserving information via flags.
- Save the cleaned dataset to `/data_processed/` for feature engineering.

Throughout, we emphasise reproducible, pipeline-friendly code that can be
moved into scripts later as part of an MLOps workflow.


---
# 1. Imports, Global Configuration & Styling


We begin by importing the core scientific Python stack and configuring:

- Paths for raw and processed data.
- A consistent plotting style.
- Helper functions to pretty-print DataFrames for easier inspection.

These utilities will be reused across the project.

In [49]:
from __future__ import annotations

import math
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google.colab import drive  # type: ignore
from scipy import stats
from sklearn.ensemble import ExtraTreesRegressor, IsolationForest
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer

sns.set_theme(style="whitegrid", context="notebook")

# ---- Drive mount (Colab) ----
drive.mount("/content/drive")

RAW_DATA_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/DOAA/data_raw/usa_real_estate.csv"
)
CLEAN_DATA_PARQUET = (
    "/content/drive/MyDrive/Colab Notebooks/DOAA/data_processed/"
    "usa_real_estate_clean.parquet"
)
CLEAN_DATA_CSV = (
    "/content/drive/MyDrive/Colab Notebooks/DOAA/data_processed/"
    "usa_real_estate_clean.csv"
)

RANDOM_STATE = 42
MAX_ROWS_FOR_FIT = 100_000  # for model-based imputers / Isolation Forest
TARGET_COL = "price"

def pretty_df(
    df: pd.DataFrame,
    precision: int = 3,
) -> pd.io.formats.style.Styler:
    """Return a beautified DataFrame Styler with borders and readable styling.

    This helper is mainly for exploratory data analysis, to make tabular
    outputs easier to read in Jupyter/Colab.

    Args:
        df: The DataFrame to beautify.
        precision: Number of decimal places for numeric values.

    Returns:
        A Styler object with table styling applied.
    """
    styler = (
        df.style.set_properties(
            **{
                "border": "1px solid #444",
                "padding": "6px",
                "color": "#111111",
                "font-size": "13px",
            }
        )
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#222"),
                        ("color", "white"),
                        ("font-weight", "bold"),
                        ("border", "1px solid #444"),
                        ("padding", "8px"),
                    ],
                },
                {
                    "selector": "tbody tr:nth-child(odd)",
                    "props": [("background-color", "#f5f5f5")],
                },
                {
                    "selector": "tbody tr:nth-child(even)",
                    "props": [("background-color", "#ffffff")],
                },
            ]
        )
        .format(precision=precision)
    )
    return styler


def styled_missing(df: pd.DataFrame) -> pd.io.formats.style.Styler:
    """Style a missingness summary table with red/green highlights.

    The input DataFrame is expected to contain at least the columns
    ``"n_missing"`` and ``"pct_missing"``. Cells in those columns will be
    highlighted:
      * Red for features with missing values.
      * Green for fully complete features.

    Args:
        df: Input DataFrame containing missingness metrics.

    Returns:
        A Styler object with conditional formatting applied to missingness
        columns.
    """

    def highlight_missing(row: pd.Series) -> List[str]:
        """Row-wise styling rule for missingness columns."""
        n_missing = row.get("n_missing", None)

        if pd.notna(n_missing) and n_missing > 0:
            color = (
                "background-color: #ffcccc;"
                "color: #660000;"
                "font-weight: bold;"
            )
        else:
            color = (
                "background-color: #ccffcc;"
                "color: #004d00;"
                "font-weight: bold;"
            )

        return [
            color if col in {"n_missing", "pct_missing"} else ""
            for col in row.index
        ]

    return pretty_df(df).apply(highlight_missing, axis=1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


With this setup, we have a consistent environment (paths, random seed) and
styling helpers that make subsequent structural and missingness tables much
easier to read during review and debugging.

---
# 2. Load Raw Data & Structural Overview

We now load the raw CSV into `df_full` and compute a structural summary:

- Data type per column.
- Number of unique values.
- Number and percentage of missing values.

This mirrors 01_eda but is now treated as the *baseline* against which we will
compare the cleaned dataset.

In [50]:
def load_raw_dataset(path: str = RAW_DATA_PATH) -> pd.DataFrame:
    """Load the raw USA real estate dataset from disk.

    Args:
        path: Path to the raw CSV file.

    Returns:
        DataFrame containing the raw dataset.
    """
    df = pd.read_csv(path)
    print(f"[INFO] Loaded dataset from {path!r} with shape {df.shape}.")
    return df


def summarize_structure(df: pd.DataFrame) -> pd.DataFrame:
    """Summarise dtypes, unique counts, and missingness per column.

    The returned DataFrame is sorted in descending order of percentage
    missingness to surface the most problematic features first.

    Args:
        df: Input dataset.

    Returns:
        DataFrame with one row per column and the following fields:
        - 'dtype': Data type of the column.
        - 'n_unique': Number of unique non-null values.
        - 'n_missing': Number of missing values.
        - 'pct_missing': Percentage of missing values.
    """
    summary = pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "n_unique": df.nunique(dropna=True),
            "n_missing": df.isna().sum(),
        }
    )
    summary["pct_missing"] = summary["n_missing"] / len(df) * 100.0
    summary = summary.sort_values("pct_missing", ascending=False)
    return summary


try:
    df_full  # type: ignore[name-defined]
    print(f"[INFO] Using existing df_full with shape {df_full.shape}.")
except NameError:
    df_full = load_raw_dataset()

structure_before = summarize_structure(df_full)

display(styled_missing(structure_before.head(20)))

[INFO] Using existing df_full with shape (2226382, 12).


,dtype,n_unique,n_missing,pct_missing
prev_sold_date,object,14954,734297,32.982
house_size,float64,12061,568484,25.534
bath,float64,86,511771,22.987
bed,float64,99,481317,21.619
acre_lot,float64,16057,325589,14.624
street,float64,2001358,10866,0.488
brokered_by,float64,110143,4533,0.204
price,float64,102137,1541,0.069
city,object,20098,1407,0.063
zip_code,float64,30334,299,0.013


From the structural summary above, we can clearly see which columns suffer the
heaviest missingness (for example `prev_sold_date`, `house_size`, `bath`,
`bed`) and the current dtypes. These insights drive the cleaning plan in the
next sections.

---
# 3. Duplicate Analysis

Before initiating data cleaning workflows, it is crucial to assess the extent of duplicated rows within the dataset. Duplicate entries can distort model training, bias summary statistics, and introduce structural noise into downstream feature engineering.

The table below reports:

- **Number of duplicated rows (`n_duplicates`)**
- **Percentage of duplicated rows (`pct_duplicates`)**

Rows are highlighted in **red** if duplicates are present, indicating the need for corrective action before modelling. A green highlight indicates an absence of duplicate entries.

In [51]:
def detect_column_types(df: pd.DataFrame) -> Dict[str, List[str]]:
    """Detect numeric, categorical, and datetime-like columns.

    Args:
        df: Input dataset.

    Returns:
        Dictionary with keys:
            - 'numeric': List of numeric columns.
            - 'categorical': List of non-numeric, non-date columns.
            - 'datetime': List of datetime-like columns.
    """
    numeric = df.select_dtypes(include=[np.number]).columns.tolist()

    datetime_like = [
        col for col in df.columns
        if "date" in col.lower() or np.issubdtype(df[col].dtype, np.datetime64)
    ]

    categorical = [
        col for col in df.columns
        if col not in numeric and col not in datetime_like
    ]

    return {
        "numeric": numeric,
        "categorical": categorical,
        "datetime": datetime_like,
    }


def duplicated_profile(df: pd.DataFrame) -> pd.DataFrame:
    """Compute duplicated row counts and percentages.

    Args:
        df: Input dataset.

    Returns:
        DataFrame containing:
            - n_duplicates: Number of duplicated rows.
            - pct_duplicates: Percentage of duplicated rows.
    """
    n_dupes = df.duplicated().sum()
    pct_dupes = (n_dupes / len(df)) * 100.0

    out = pd.DataFrame(
        {
            "n_duplicates": [n_dupes],
            "pct_duplicates": [pct_dupes],
        },
        index=["dataset_level"],
    )
    return out


# ---- Execute profiles ----
column_types = detect_column_types(df_full)
duplicate_before = duplicated_profile(df_full)

display(
    pretty_df(duplicate_before).apply(
        lambda row: [
            (
                "background-color: #ffcccc; color: #660000; font-weight: bold;"
                if row["n_duplicates"] > 0
                else "background-color: #ccffcc; color: #004d00; "
                     "font-weight: bold;"
            ),
            (
                "background-color: #ffcccc; color: #660000; font-weight: bold;"
                if row["n_duplicates"] > 0
                else "background-color: #ccffcc; color: #004d00; "
                     "font-weight: bold;"
            ),
        ],
        axis=1,
    )
)

,n_duplicates,pct_duplicates
dataset_level,0,0.000


Following the duplicate-removal process, the dataset was re-evaluated to confirm that redundant rows were successfully eliminated. This verification step ensures the structural integrity of the cleaned dataset and prevents accidental re-introduction of duplicates during subsequent transformations.

---
# 4. Duplicate Handling


Before we start imputing and flagging outliers, we should ensure that the
dataset does not contain exact duplicate records. Keeping duplicates would:

- Overweight certain observations during model training.
- Distort distributions and outlier detection.
- Potentially bias evaluation metrics.

We perform a simple but explicit duplicate check, then drop duplicates while
recording how many rows were removed.

In [52]:
def drop_duplicates_safely(
    df: pd.DataFrame,
    subset: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, int]:
    """Remove duplicate rows safely and report the number removed.

    This helper provides a controlled approach for deduplication during
    cleaning pipelines. It preserves the first occurrence of each record
    (based on the specified subset of columns) and resets the index
    afterwards for consistency.

    Args:
        df: Input DataFrame to deduplicate.
        subset: Optional list of column names to use for duplicate
            detection. If ``None``, all columns are considered.

    Returns:
        A tuple containing:
            - A new DataFrame with duplicates removed.
            - The integer count of duplicate rows that were removed.
    """
    n_before = len(df)
    df_clean = (
        df.drop_duplicates(subset=subset, keep="first")
        .reset_index(drop=True)
    )
    n_removed = n_before - len(df_clean)

    return df_clean, n_removed

This helper encapsulates duplicate logic and returns both the cleaned
DataFrame and the number of rows removed. By default it uses *all* columns,
but we can pass a subset later if the business logic defines a different
notion of duplicate.

---
# 5. Auto-Selecting Imputation Strategy

We now define a rule-based selector that decides how each feature should be
imputed based on:

- Data type (`dtype`)
- Percentage of missing values, $\text{pct\_missing}$
- Number of unique values, $n\_{\text{unique}}$

High-level rules:

- **Numeric**  
  - Low missingness (≤ 5%) → median imputation.  
  - Moderate missingness (≤ 30%) → IterativeImputer with
    ExtraTreesRegressor.  
  - Very high missingness  → KNNImputer across the numeric block.
- **Categorical**  
  - Low cardinality utilises mode imputation.  
  - High cardinality / noisy utilises `"unknown"` token.

This keeps the notebook data-driven instead of hard-coding each column.

In [53]:
def choose_imputation_strategy(
    col_name: str,
    dtype: str,
    pct_missing: float,
    n_unique: int,
) -> str:
    """Select an imputation strategy based on feature metadata.

    Special rules:
        - If the target column ``"price"`` contains missing values,
          the column will be dropped entirely (strategy: ``"drop_column"``).
        - If a column name contains ``"date"`` and has missing values,
          missing entries will be filled with the string ``"unknown"``
          (strategy: ``"unknown_date"``).

    General strategy options:
        - "skip": No missing values → do nothing.
        - "median": Numeric median (low missingness).
        - "iterative_et": IterativeImputer + ExtraTrees (moderate missingness).
        - "knn": KNNImputer (high numeric missingness).
        - "mode": Categorical mode (low/moderate missingness).
        - "unknown_token": Fill with "unknown" category.
        - "unknown_date": Fill date-like column with "unknown" (as string).
        - "drop_column": Drop the column entirely.

    Args:
        col_name: Name of the column.
        dtype: Column dtype as string (e.g. "float64", "int64", "object").
        pct_missing: Percentage missing (0–100).
        n_unique: Number of unique non-null values.

    Returns:
        The name of the selected imputation strategy.
    """
    col_lower = col_name.lower()

    # --- Special rule: price should not have missing values ---
    if col_lower == "price" and pct_missing > 0:
        return "skip"

    # --- Special rule: any date-like column with missing values ---
    if "date" in col_lower:
        if pct_missing > 0:
            return "unknown_date"
        return "skip"

    # --- No missing values ---
    if pct_missing == 0:
        return "skip"

    # --- Numeric features ---
    if dtype.startswith(("float", "int")):
        if pct_missing <= 5:
            return "median"
        if pct_missing <= 30:
            return "iterative_et"
        return "knn"

    # --- Categorical / object-like features ---
    if n_unique <= 50 and pct_missing <= 40:
        return "mode"

    return "unknown_token"

The table above documents the planned imputation strategy for each feature,
driven entirely by the observed missingness and feature type. This plan is
recorded as metadata and will be executed programmatically in the main
cleaning pipeline.

---
# 6. Univariate Outliers Addressment

For each numeric feature, we automatically select an appropriate
univariate outlier method:

---
**Interquartile Range (IQR) Rule**

Purpose: Flags Points far from the Central 50% of the data

$$\text{IQR} = Q_3 - Q_1$$

$$\text{Lower Bound} = Q_1 - k \times \text{IQR}$$

$$\text{Upper Bound} = Q_3 + k \times \text{IQR}$$

**Legend:**
- $Q_1$ = 25th Percentile
- $Q_3$ = 75th Percentile
- $\text{IQR}$ = Interquartile range
- $k$ = IQR Multiplier (commonly 1.5 or 3.0)

---
**Median Absolute Deviation (MAD)**

Purpose: Flags Points far from the Central 50% of the data

$$\text{MAD} = \text{median}\left(|x_i - \tilde{x}|\right)$$

$$z_i = \frac{0.6745 \times (x_i - \tilde{x})}{\text{MAD}}$$

$$\text{Outlier if } |z_i| > T_{\text{MAD}}$$

**Legend:**
- $Q_1$ = 25th Percentile
- $Q_3$ = 75th Percentile
- $\text{IQR}$ = Interquartile range
- $k$ = IQR Multiplier (commonly 1.5 or 3.0)

We pick the method based on skewness and kurtosis of each feature.

In [54]:
def choose_univariate_method(series: pd.Series) -> str:
    """Select a univariate outlier detection method based on distribution shape.

    Method selection heuristic:
        - If the distribution is approximately Gaussian (low skew and kurtosis),
          use ``"zscore"``.
        - If moderately skewed, use ``"iqr"``.
        - If heavily skewed or heavy-tailed, use ``"mad"``.

    Args:
        series: Numeric pandas Series to evaluate.

    Returns:
        A string representing the chosen method:
        ``"zscore"``, ``"iqr"``, or ``"mad"``.
    """
    x = (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if x.empty:
        return "iqr"

    skew_val = stats.skew(x)
    kurt_val = stats.kurtosis(x, fisher=False)

    # Approximately Gaussian distribution
    if abs(skew_val) < 0.5 and abs(kurt_val - 3.0) < 1.0:
        return "zscore"

    # Moderately skewed
    if abs(skew_val) < 1.0:
        return "iqr"

    # Heavy-tailed / highly skewed
    return "mad"


def flag_univariate_outliers(
    series: pd.Series,
    method: Optional[str] = None,
    z_thresh: float = 3.5,
    iqr_factor: float = 1.5,
    mad_factor: float = 3.5,
) -> pd.Series:
    """Flag univariate outliers in a numeric Series using a selected method.

    Supported methods:
        - ``"zscore"``: Standardised Z-score thresholding.
        - ``"iqr"``: Tukey's rule using the interquartile range.
        - ``"mad"``: Median Absolute Deviation (robust to heavy tails).

    Args:
        series: Numeric Series on which to perform outlier detection.
        method: Optional override for method selection. If ``None``,
            ``choose_univariate_method`` is used.
        z_thresh: Threshold for Z-score outliers.
        iqr_factor: IQR multiplier determining outlier cutoffs.
        mad_factor: Threshold for modified Z-score using MAD.

    Returns:
        A boolean Series where ``True`` indicates an outlier.
    """
    x = (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    # Auto-select method if none provided
    if method is None:
        method = choose_univariate_method(x)

    # Z-score method
    if method == "zscore":
        z_values = stats.zscore(x, nan_policy="omit")
        return pd.Series(np.abs(z_values) > z_thresh, index=series.index)

    # Interquartile Range method
    if method == "iqr":
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - iqr_factor * iqr
        upper_bound = q3 + iqr_factor * iqr
        return (x < lower_bound) | (x > upper_bound)

    # Median Absolute Deviation (MAD) — default
    median = x.median()
    mad = np.median(np.abs(x - median))

    # Avoid division by zero
    if mad == 0:
        return pd.Series(False, index=series.index)

    modified_z = 0.6745 * (x - median) / mad
    return pd.Series(np.abs(modified_z) > mad_factor, index=series.index)

This utility allows the notebook to automatically decide how to flag extreme
values per feature based on its empirical distribution, while still exposing
clear hyperparameters (thresholds, factors) that can be tuned if needed.

---
# 7. Multivariate Outlier Addressment

Beyond univariate rules, we use **Isolation Forest** to detect rows that are
anomalous across multiple features simultaneously.

Conceptually, Isolation Forest repeatedly partitions the feature space;
outliers tend to be isolated in fewer splits, leading to lower anomaly
scores.

We set a small contamination rate (e.g. 1%) to flag a small proportion of
rows as multivariate outliers.

In [55]:
def detect_multivariate_outliers_iforest(
    df: pd.DataFrame,
    numeric_cols: List[str],
    contamination: float = 0.01,
) -> pd.Series:
    """Detect multivariate outliers using an IsolationForest model.

    This function applies IsolationForest over a numeric feature block to
    identify jointly anomalous rows. It performs a simple median imputation
    for numeric features before fitting and predicting, and subsamples the
    dataset when it exceeds ``MAX_ROWS_FOR_FIT`` to keep training efficient.

    Args:
        df: Input DataFrame containing the features.
        numeric_cols: List of numeric column names to use as features.
        contamination: Estimated proportion of outliers in the data.

    Returns:
        A boolean Series indexed like ``df``, where ``True`` indicates
        a row flagged as a multivariate outlier.
    """
    if not numeric_cols:
        return pd.Series(False, index=df.index)

    # Work on a numeric-only copy
    X = df[numeric_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)

    # Median imputation per column for model fitting
    X_med = X.fillna(X.median(numeric_only=True))

    # Optional subsampling to avoid excessive fit cost
    if len(X_med) > MAX_ROWS_FOR_FIT:
        X_fit = X_med.sample(
            n=MAX_ROWS_FOR_FIT,
            random_state=RANDOM_STATE,
        )
    else:
        X_fit = X_med

    model = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_fit)

    # Predictions on full (imputed) numeric block: -1 = outlier, 1 = inlier
    preds = model.predict(X_med)
    outlier_flags = preds == -1

    print(
        f"[INFO] IsolationForest flagged {outlier_flags.sum():,} "
        f"rows as multivariate outliers "
        f"({outlier_flags.mean() * 100:.3f}%)."
    )

    return pd.Series(outlier_flags, index=df.index)

With this function, we can flag a small fraction of rows as globally
suspicious based on the joint behaviour of all numeric features, not just
one at a time. We keep the flags for transparency and can optionally drop
those rows if the project requires a “clean-only” training set.

---
# 8. Missing Values Imputation


For numeric features with moderate to high missingness, we prefer
model-based imputation over simple rules:

- **IterativeImputer + ExtraTreesRegressor**  
  Each feature is modelled as a function of the others using tree ensembles,
  iteratively refining imputations.

- **KNNImputer**  
  For very high missingness, we fall back to $k$-nearest neighbours on the
  numeric block.

This allows more expressive imputations while still being practical on the
dataset size (we subsample for fitting when necessary).

In [56]:
def fit_iterative_imputer_et(
    df: pd.DataFrame,
    numeric_cols: List[str],
    max_rows: int = MAX_ROWS_FOR_FIT,
) -> IterativeImputer:
    """Fit an IterativeImputer with ExtraTreesRegressor on numeric features.

    This helper prepares a robust, model-based imputer for numeric
    data. It handles infinities, performs optional row subsampling
    for memory/runtime control, and configures ExtraTrees with a
    reasonable ensemble size.

    Args:
        df: Input dataset containing numeric columns.
        numeric_cols: List of numeric column names to impute.
        max_rows: Maximum number of rows used to fit the imputer.

    Returns:
        Fitted IterativeImputer instance using ExtraTreesRegressor.

    Raises:
        ValueError: If no numeric columns are provided.
    """
    if not numeric_cols:
        raise ValueError("No numeric columns provided for IterativeImputer.")

    X = df[numeric_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)

    if len(X) > max_rows:
        X_fit = X.sample(n=max_rows, random_state=RANDOM_STATE)
    else:
        X_fit = X

    et_reg = ExtraTreesRegressor(
        n_estimators=50,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    imputer = IterativeImputer(
        estimator=et_reg,
        max_iter=40,
        random_state=RANDOM_STATE,
        initial_strategy="median",
    )
    imputer.fit(X_fit)

    return imputer


def apply_iterative_imputer_et(
    df: pd.DataFrame,
    numeric_cols: List[str],
    imputer: IterativeImputer,
) -> pd.DataFrame:
    """Apply a fitted IterativeImputer to numeric columns and ensure
    discrete fields like bed/bath are rounded to whole numbers.

    This function transforms numeric columns using the provided imputer,
    preserves DataFrame structure, and applies rounding rules for
    discrete home-attribute features such as bed and bath counts.

    Args:
        df: Dataset to transform.
        numeric_cols: List of numeric column names to impute.
        imputer: Fitted IterativeImputer instance.

    Returns:
        DataFrame with numeric columns imputed and corrected.
    """
    X = df[numeric_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)

    X_imputed = imputer.transform(X)

    df_out = df.copy()
    df_out[numeric_cols] = X_imputed

    # --- Enforce whole-number rounding for discrete count features ---
    for col in ["bed", "bath"]:
        if col in df_out.columns:
            df_out[col] = (
                df_out[col]
                .round(0)        # Round to nearest whole
                .clip(lower=0)   # No negative room counts
                .astype("int16")
            )

    return df_out

These helpers let us learn imputation patterns on a sampled subset and then
apply them back to the full dataset, balancing statistical power with
computational efficiency for large-rowcount data.

---
# 9. Full Cleaning Pipeline

We now bring everything together in a reusable `clean_dataset()` function.

High-level steps:

1. Derive imputation plan from structure.
2. Create missingness indicator flags.
3. Normalise dtypes (numeric, categorical, datetime).
4. Apply numeric imputation:
   - median / IterativeImputer (ExtraTrees) / KNN based on plan.
5. Apply categorical imputation (mode / `"unknown"`).
6. Flag univariate outliers and optionally cap them.
7. Detect multivariate outliers with Isolation Forest.
8. Optionally drop outlier rows.
9. Return the cleaned DataFrame and a rich metadata dictionary.

In [57]:
def clean_dataset(
    df: pd.DataFrame,
    remove_univariate_outliers: bool = False,
    remove_multivar_outliers: bool = False,
    cap_univariate_outliers: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, Dict]]:
    """Clean the USA real estate dataset using advanced methods.

    Processing steps:
        0) Drop duplicate rows.
        0.25) Clean target 'price' into a proper numeric column.
        0.5) Enforce target rules:
                • Never drop column 'price'.
                • Drop rows where 'price' is missing (after cleaning).
        1) Derive imputation strategy per column.
        2) Drop columns marked as "drop_column" (except 'price').
        3) Create missingness indicator flags.
        4) Normalise dtypes (dates, categoricals, numerics).
        5) Apply numeric imputation (median / Iterative + ExtraTrees / KNN).
        6) Apply categorical + date imputation (mode / 'unknown').
        7) Detect and flag univariate and multivariate outliers on key
           numeric columns (price, house_size, bed, bath, acre_lot).
        8) Remove only impossible/corrupted values and optionally
           drop/cap outliers.
        9) Enforce discrete counts for bed and bath, then return
           cleaned DataFrame and metadata.

    Args:
        df: Raw input dataset.
        remove_univariate_outliers: If True, drop rows flagged as
            univariate outliers on the selected numeric columns.
        remove_multivar_outliers: If True, drop rows flagged as
            multivariate outliers by IsolationForest.
        cap_univariate_outliers: If True, clip univariate outliers instead
            of just flagging.

    Returns:
        Tuple of:
            * cleaned: Cleaned DataFrame.
            * meta: Dictionary containing strategies, diagnostics, and
              summary statistics for the cleaning process.
    """
    meta: Dict[str, Dict] = {}

    # --- 0) Drop duplicates upfront ---
    df_dedup, n_dupes_removed = drop_duplicates_safely(df)
    meta["duplicates"] = {
        "removed": int(n_dupes_removed),
        "rows_before": int(len(df)),
        "rows_after": int(len(df_dedup)),
    }

    cleaned = df_dedup.copy()
    n_rows_before = len(cleaned)

    # --- 0.25) Clean target 'price' into a proper numeric column ---
    if TARGET_COL not in cleaned.columns:
        raise KeyError(f"Target column '{TARGET_COL}' not found in input dataset.")

    # Convert price to string, strip whitespace, remove non-numeric chars except dot/minus
    cleaned[TARGET_COL] = (
        cleaned[TARGET_COL]
        .astype("string")
        .str.strip()
        .str.replace(r"[^\d\.\-]", "", regex=True)  # keep only digits, dot, minus
        .replace("", np.nan)
        .astype("float64")
    )

    # --- 0.5) Target enforcement: drop rows with missing price, keep column ---
    n_missing_price_initial = int(cleaned[TARGET_COL].isna().sum())
    if n_missing_price_initial > 0:
        print(
            f"[INFO] Dropping {n_missing_price_initial} rows with missing "
            f"{TARGET_COL} after initial price cleaning."
        )
        cleaned = cleaned.loc[cleaned[TARGET_COL].notna()].copy()
    meta["target_price_missing_rows_dropped_initial"] = n_missing_price_initial

    # --- Type & missingness info (after dropping missing price) ---
    struct = summarize_structure(cleaned)
    col_types = detect_column_types(cleaned)
    numeric_cols = col_types["numeric"]
    categorical_cols = col_types["categorical"]
    datetime_cols = col_types["datetime"]

    # --- 1) Imputation plan (from struct) ---
    plan_rows: List[Dict[str, object]] = []
    for col, row in struct.iterrows():
        strategy = choose_imputation_strategy(
            col_name=col,
            dtype=row["dtype"],
            pct_missing=float(row["pct_missing"]),
            n_unique=int(row["n_unique"]),
        )
        plan_rows.append(
            {
                "column": col,
                "dtype": row["dtype"],
                "pct_missing": float(row["pct_missing"]),
                "n_missing": int(row["n_missing"]),
                "strategy": strategy,
            }
        )

    impute_plan = pd.DataFrame(plan_rows).set_index("column")
    meta["impute_plan"] = impute_plan.to_dict(orient="index")

    # --- 2) Drop columns marked as 'drop_column' (never drop TARGET_COL) ---
    drop_cols = impute_plan[impute_plan["strategy"] == "drop_column"].index.tolist()
    if TARGET_COL in drop_cols:
        drop_cols = [c for c in drop_cols if c != TARGET_COL]

    if drop_cols:
        cleaned = cleaned.drop(columns=drop_cols)
        meta["dropped_columns"] = {
            "columns": drop_cols,
            "reason": "imputation_strategy=drop_column",
        }
        numeric_cols = [c for c in numeric_cols if c not in drop_cols]
        categorical_cols = [c for c in categorical_cols if c not in drop_cols]
        datetime_cols = [c for c in datetime_cols if c not in drop_cols]

    # --- 2.5) Date columns with 'unknown_date' → treat as categorical ---
    unknown_date_cols = impute_plan[
        impute_plan["strategy"] == "unknown_date"
    ].index.tolist()
    unknown_date_cols = [c for c in unknown_date_cols if c in cleaned.columns]

    datetime_cols = [c for c in datetime_cols if c not in unknown_date_cols]
    categorical_cols = categorical_cols + unknown_date_cols

    # --- 3) Missingness flags ---
    struct_after_drop = summarize_structure(cleaned)
    for col in cleaned.columns:
        if struct_after_drop.loc[col, "n_missing"] > 0:
            flag_col = f"{col}_was_missing"
            cleaned[flag_col] = cleaned[col].isna().astype("int8")

    # --- 4) Normalise dtypes ---

    # Datetime columns
    for col in datetime_cols:
        cleaned[col] = pd.to_datetime(
            cleaned[col],
            errors="coerce",
            infer_datetime_format=True,
        )

    # Categorical → string (includes unknown_date_cols)
    for col in categorical_cols:
        cleaned[col] = cleaned[col].astype("string")

    # Numeric coercion (price is already numeric but harmless to include)
    for col in numeric_cols:
        cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

    # NOTE: We DO NOT drop on price again here.
    # Any NaNs in other numeric columns will be handled by imputation.

    # --- 5) Numeric block imputation ---

    numeric_advanced = [
        col
        for col in numeric_cols
        if col in impute_plan.index
        and impute_plan.loc[col, "strategy"] == "iterative_et"
        and impute_plan.loc[col, "n_missing"] > 0
    ]
    numeric_knn = [
        col
        for col in numeric_cols
        if col in impute_plan.index
        and impute_plan.loc[col, "strategy"] == "knn"
        and impute_plan.loc[col, "n_missing"] > 0
    ]
    numeric_simple = [
        col
        for col in numeric_cols
        if col in impute_plan.index
        and impute_plan.loc[col, "strategy"] == "median"
        and impute_plan.loc[col, "n_missing"] > 0
    ]

    # Ensure target is never imputed even if plan mislabels it
    if TARGET_COL in numeric_advanced:
        numeric_advanced.remove(TARGET_COL)
    if TARGET_COL in numeric_knn:
        numeric_knn.remove(TARGET_COL)
    if TARGET_COL in numeric_simple:
        numeric_simple.remove(TARGET_COL)

    # 5a) Simple median
    meta["imputation_numeric_simple"] = {}
    for col in numeric_simple:
        median_val = cleaned[col].median()
        cleaned[col] = cleaned[col].fillna(median_val)
        meta["imputation_numeric_simple"][col] = {
            "strategy": "median",
            "value": float(median_val),
        }

    # 5b) IterativeImputer + ExtraTrees
    if numeric_advanced:
        print(f"[INFO] IterativeImputer on columns: {numeric_advanced}")
        imp_et = fit_iterative_imputer_et(cleaned, numeric_advanced)
        cleaned = apply_iterative_imputer_et(
            cleaned,
            numeric_cols=numeric_advanced,
            imputer=imp_et,
        )
        meta["imputation_numeric_iterative_et"] = {
            "columns": numeric_advanced,
            "estimator": "ExtraTreesRegressor",
        }

    # 5c) KNNImputer
    if numeric_knn:
        print(f"[INFO] KNNImputer on columns: {numeric_knn}")
        knn = KNNImputer(n_neighbors=5, weights="distance")
        X_knn = cleaned[numeric_knn].copy()
        X_knn = X_knn.replace([np.inf, -np.inf], np.nan)
        X_knn_imputed = knn.fit_transform(X_knn)
        cleaned[numeric_knn] = X_knn_imputed
        meta["imputation_numeric_knn"] = {
            "columns": numeric_knn,
            "n_neighbors": 5,
        }

    # --- 6) Categorical + date imputation ---
    meta["imputation_categorical"] = {}
    for col in categorical_cols:
        if col not in impute_plan.index:
            continue

        strat = impute_plan.loc[col, "strategy"]

        if strat == "skip":
            continue

        if strat == "mode":
            mode_val = cleaned[col].mode(dropna=True)
            fill_val = mode_val.iloc[0] if not mode_val.empty else "unknown"
            cleaned[col] = cleaned[col].fillna(fill_val)
            meta["imputation_categorical"][col] = {
                "strategy": "mode",
                "value": str(fill_val),
            }
        elif strat in {"unknown_token", "unknown_date"}:
            cleaned[col] = cleaned[col].fillna("unknown")
            meta["imputation_categorical"][col] = {
                "strategy": strat,
                "value": "unknown",
            }

    # --- 7) Advanced outlier handling on key numeric columns ---

    target_outlier_cols = [
        c
        for c in ["price", "house_size", "bed", "bath", "acre_lot"]
        if c in numeric_cols
    ]

    meta["log_transform"] = {}
    meta["univariate_outliers"] = {}
    meta["multivariate_outliers"] = {}
    meta["contextual_outliers"] = {}

    # 7.1) Log-transform heavily skewed columns (keep original)
    for col in target_outlier_cols:
        series = cleaned[col]
        skew_val = series.skew()
        if abs(skew_val) > 1.0:
            cleaned[f"{col}_log"] = np.log1p(series.clip(lower=0))
            meta["log_transform"][col] = float(skew_val)

    # 7.2) Univariate outlier flagging
    for col in target_outlier_cols:
        series = cleaned[col]
        method = choose_univariate_method(series)
        flags = flag_univariate_outliers(series, method=method)
        cleaned[f"{col}__is_uni_outlier"] = flags.astype("int8")
        outlier_rate = float(flags.mean() * 100.0)
        meta["univariate_outliers"][col] = {
            "method": method,
            "outlier_rate_pct": outlier_rate,
        }

        # Optional clipping (IQR-based)
        if cap_univariate_outliers and flags.any():
            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            cleaned[col] = series.clip(lower=lower, upper=upper)

    # 7.3) Multivariate outliers via IsolationForest
    iforest_cols = [
        c for c in ["price", "house_size", "bed", "bath", "acre_lot"]
        if c in numeric_cols
    ]

    multi_flags: Optional[pd.Series] = None
    if iforest_cols:
        multi_flags = detect_multivariate_outliers_iforest(
            cleaned,
            numeric_cols=iforest_cols,
            contamination=0.01,
        )
        cleaned["is_multivar_outlier"] = multi_flags.astype("int8")
        meta["multivariate_outliers"] = {
            "contamination": 0.01,
            "columns": iforest_cols,
            "outlier_rate_pct": float(multi_flags.mean() * 100.0),
        }

    # 7.4) Contextual peer comparison for price (city + zip_code)
    if all(c in cleaned.columns for c in ["price", "city", "zip_code"]):
        peer_mean = (
            cleaned.groupby(["city", "zip_code"])["price"].transform("mean")
        )
        peer_std = (
            cleaned.groupby(["city", "zip_code"])["price"]
            .transform("std")
            .fillna(1.0)
        )
        cleaned["price_peer_z"] = (cleaned["price"] - peer_mean) / peer_std
        cleaned["is_price_context_outlier"] = (
            cleaned["price_peer_z"].abs() > 3
        ).astype("int8")

        meta["contextual_outliers"] = {
            "rule": "abs(price_peer_z) > 3 within (city, zip_code)",
            "outlier_rate_pct": float(
                (cleaned["is_price_context_outlier"] == 1).mean() * 100.0
            ),
        }

    # --- 8) Remove impossible / corrupted values + optional outlier row drops ---

    impossible_mask = pd.Series(False, index=cleaned.index)

    if "price" in cleaned.columns:
        impossible_mask |= cleaned["price"] < 0
    if "house_size" in cleaned.columns:
        impossible_mask |= cleaned["house_size"] <= 0
    if "bed" in cleaned.columns:
        impossible_mask |= cleaned["bed"] < 0
    if "bath" in cleaned.columns:
        impossible_mask |= cleaned["bath"] < 0
    if "acre_lot" in cleaned.columns:
        impossible_mask |= cleaned["acre_lot"] < 0

    n_impossible = int(impossible_mask.sum())
    if n_impossible > 0:
        cleaned = cleaned.loc[~impossible_mask].copy()
    meta["rows_removed_impossible"] = n_impossible

    # 8.2) Optional: row drops based on univariate flags
    if remove_univariate_outliers and target_outlier_cols:
        uni_flag_cols = [
            f"{c}__is_uni_outlier" for c in target_outlier_cols
            if f"{c}__is_uni_outlier" in cleaned.columns
        ]
        uni_flags = cleaned[uni_flag_cols].max(axis=1).astype(bool)
        mask_keep_uni = ~uni_flags
        n_drop_uni = int((~mask_keep_uni).sum())
        if n_drop_uni > 0:
            cleaned = cleaned.loc[mask_keep_uni].copy()
        meta["rows_removed_univariate"] = n_drop_uni
        print(
            f"[INFO] Removed {n_drop_uni:,} rows due to univariate outliers "
            f"({n_drop_uni / n_rows_before * 100:.3f}%)."
        )

    # 8.3) Optional: row drops based on multivariate flags
    if (
        remove_multivar_outliers
        and iforest_cols
        and "is_multivar_outlier" in cleaned.columns
    ):
        multi_flags_aligned = cleaned["is_multivar_outlier"] == 1
        mask_keep_multi = ~multi_flags_aligned
        n_drop_multi = int((~mask_keep_multi).sum())
        if n_drop_multi > 0:
            cleaned = cleaned.loc[mask_keep_multi].copy()
        meta["rows_removed_multivariate"] = n_drop_multi
        print(
            f"[INFO] Removed {n_drop_multi:,} rows due to multivariate "
            f"outliers ({n_drop_multi / n_rows_before * 100:.3f}%)."
        )

    # --- 9) Enforce discrete counts for bed and bath ---
    for col in ["bed", "bath"]:
        if col in cleaned.columns:
            cleaned[col] = (
                cleaned[col]
                .round(0)
                .clip(lower=0)
                .astype("int16")
            )

    # Final sanity check: price must have no NaNs
    if TARGET_COL in cleaned.columns:
        assert cleaned[TARGET_COL].isna().sum() == 0, (
            "Invariant broken: price has NaNs at the end of clean_dataset()."
        )

    print(
        f"[INFO] Cleaning finished. Rows before: {n_rows_before:,} — "
        f"after: {len(cleaned):,}."
    )

    return cleaned, meta

The `clean_dataset()` function encapsulates the entire logic in a single,
reusable pipeline. It also returns a `meta` dictionary capturing imputation
plans, outlier statistics, and any row removals, which is useful both for
auditing and for writing the report later.

---
# 10. Cleaning Execution & Comparison

We now run the cleaning pipeline on the full dataset and compare structural
statistics pre-cleaning and post-cleaning.

In [62]:
df_clean, cleaning_meta = clean_dataset(
    df_full,
    remove_univariate_outliers=False,
    remove_multivar_outliers=False,
    cap_univariate_outliers=True,
)

df_clean = df_clean.drop(
    columns=["price_peer_z", "is_price_context_outlier"],
    errors="ignore",
)

structure_after = summarize_structure(df_clean)

display(
    styled_missing(
        structure_after.head(20)
    )
)

impute_plan_example = (
    pd.DataFrame(cleaning_meta["impute_plan"])
    .T
    .head(10)
)

[INFO] Dropping 1541 rows with missing price after initial price cleaning.
[INFO] IterativeImputer on columns: ['bed', 'bath', 'acre_lot', 'house_size']


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


[INFO] IsolationForest flagged 22,781 rows as multivariate outliers (1.024%).
[INFO] Cleaning finished. Rows before: 2,226,382 — after: 2,224,841.


,dtype,n_unique,n_missing,pct_missing
brokered_by,float64,110121,0,0.000
status,string,3,0,0.000
price,float64,90425,0,0.000
bed,int16,5,0,0.000
bath,int16,4,0,0.000
acre_lot,float64,13272,0,0.000
street,float64,2000215,0,0.000
city,string,20090,0,0.000
state,string,56,0,0.000
zip_code,float64,30326,0,0.000


The “after” structural summary confirms that missingness has been reduced in
key columns and that dtypes remain consistent. The metadata snapshot shows
the recorded imputation strategies for selected features, which will be
helpful for documentation and explainability.

---
# 11. Sanity Checks on Core Features

To ensure that the cleaning process has not produced pathological values,
we inspect the descriptive statistics of the main numeric drivers:

- `price`
- `house_size`
- `bed`
- `bath`
- `acre_lot` (if present)

We expect to see more sensible ranges and reduced influence from extreme
outliers.

In [59]:
cols_to_check = [
    c for c in ["price", "house_size", "bed", "bath", "acre_lot"]
    if c in df_clean.columns
]

summary_frames = []

for col in cols_to_check:
    desc = df_clean[col].describe().to_frame(name=col)
    summary_frames.append(desc)

# Combine along columns
summary_df = pd.concat(summary_frames, axis=1)

display(pretty_df(summary_df))

,price,house_size,bed,bath,acre_lot
count,2224841.000,2224841.000,2224841.000,2224841.000,2224841.000
mean,403363.905,2190.964,3.520,2.545,0.706
std,314687.127,1053.055,1.177,0.948,0.790
min,0.000,4.000,2.000,1.000,0.000
25%,165000.000,1414.000,3.000,2.000,0.160
50%,325000.000,1946.460,3.000,2.000,0.280
75%,550000.000,2707.920,4.000,3.000,1.000
max,1127500.000,4648.800,6.000,4.000,2.260


The post-cleaning summaries provide a quick validation that the main
continuous features now sit on more stable ranges, with obviously impossible
extremes trimmed or capped. Any remaining unusual patterns can be handled at
the modelling or feature-engineering stage if needed.

---
# 12. Exporting Cleaned Dataset

Finally, we persist the cleaned dataset for downstream use. We save in both:

- **Parquet**: efficient binary format for Python-based pipelines.
- **CSV**: convenient for quick inspection or use in non-Python tools.

The paths follow the project's `/data_processed/` convention.

In [60]:
def save_clean_dataset(
    df: pd.DataFrame,
    parquet_path: str = CLEAN_DATA_PARQUET,
    csv_path: str = CLEAN_DATA_CSV,
) -> None:
    """Save the cleaned dataset to disk in Parquet and CSV formats.

    Args:
        df: Cleaned dataset.
        parquet_path: Output path for Parquet file.
        csv_path: Output path for CSV file.
    """
    df.to_parquet(parquet_path, index=False)
    print(f"[INFO] Saved cleaned dataset to {parquet_path!r}.")

    df.to_csv(csv_path, index=False)
    print(f"[INFO] Saved cleaned dataset to {csv_path!r}.")


save_clean_dataset(df_clean, CLEAN_DATA_PARQUET, CLEAN_DATA_CSV)

[INFO] Saved cleaned dataset to '/content/drive/MyDrive/Colab Notebooks/DOAA/data_processed/usa_real_estate_clean.parquet'.
[INFO] Saved cleaned dataset to '/content/drive/MyDrive/Colab Notebooks/DOAA/data_processed/usa_real_estate_clean.csv'.


The cleaned dataset is now available under `/data_processed/` and can be
safely used by the next notebook (`03_feature_engineering`) without
repeating any heavy cleaning logic. The cleaning pipeline can later be
refactored into a Python module and wired into a full MLOps workflow if required.